Schema

 - Matches:
     - MatchId: int
         - MatchWeek: int
         - HomeTeam: int
         - AwayTeam: int
         - MatchResult: pt.MatchResult
         - ThreadUrl: str
         - MatchTime: datetime (convert to UTC?)

 - Standings:
     - MatchId: int
         - UserName: str
             - Points: int

 - Predictions:
     - MatchId: int
         - UserName: str
             - Comment: str
             - CommentTime: datetime (convert to UTC?)


What if someone posts two comments?

{
    "MatchId": {
        "0": {
            "MatchWeek": 0,
            "HomeTeam": "Southampton",
            "
        }
    }
}

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
from icecream import ic

import predthread as pt
from predthread import fbref

from collections import defaultdict
from copy import deepcopy

In [3]:
root = Path(".").absolute()
print(root)

/Users/julianirwin/Projects/predthread/examples/southamton_24-25_33c895d4


In [4]:
club_names = pt.club_names(root)
api_keys = pt.api_keys(root)
match_metadata = pt.match_metadata(root, match_id=0)
matches_metadata = pt.matches_metadata(root)
matches_metadata_df = pt.matches_metadata_df(root)

# ic(club_names)
# ic(match_metadata)
# ic(matches_metadata)

In [11]:
# Of last completed match
match_id = 2

In [9]:
# Using matchweek, not match id
ic(fbref.match_metadata(i_matchweek=4))
pass

ic| fbref.match_metadata(i_matchweek=4): {'AwayClub': 'Manchester Utd',
                                          'AwayGoals': '',
                                          'HomeClub': 'Southampton',
                                          'HomeGoals': '',
                                          'MatchTimeUtc': '2024-09-14T12:30:00Z',
                                          'MatchWeek': 4,
                                          'ThreadUrl': ''}


In [10]:
pt.append_match_metadata(root, fbref.match_metadata(4))

In [12]:
pd.set_option("display.max_rows", None)
ps = pt.download_predictions(root, match_id)
ps.sort_values(["PredictedHomeGoals", "PredictedAwayGoals"], ascending=False)
# ps

,Comment,PredictedHomeGoals,PredictedAwayGoals
UserName,,,
nape27,3-2,3,2
flailingpariah,3-1 to Brentford. Unrelenting pessimism campai...,3,1
chebalebs,3-1,3,1
StreetSuspicious8456,3-1 saints,3,1
LegitimatePass6924,3-1,3,1
lemezier,3-1,3,1
GordonRamsayGhost,3-0,3,0
thinlike_napkins,3-0,3,0
uuuuuuuhhhhh,3-0,3,0


In [13]:
pt.summarize_predictions(root, match_id)

{'TotalPredictions': 95,
 'TotalHomeWinPredictions': 41,
 'TotalAwayWinPredictions': 30,
 'TotalDrawPredictions': 24}

In [ ]:
ps.loc["SomeGuyCalledPercy"]

In [14]:
new_standings = pt.calculate_updated_standings_after(root, match_id)
new_standings

,Points,PointsGained,Streak,Exacts,Corrects,Wrongs,MaxStreak
UserName,,,,,,,
flailingpariah,5,3,5,1,2,0,5
LegitimatePass6924,4,3,3,1,1,1,3
chebalebs,4,3,3,1,1,1,3
lemezier,4,3,3,1,1,1,3
Nixstricks,4,1,1,1,1,1,3
PM_ME_YOUR_MARS_BARS,4,0,4,1,1,0,4
halarhala,4,0,4,1,1,0,4
slugmaniac,4,0,0,1,1,1,4
StreetSuspicious8456,3,3,3,1,0,0,3


In [ ]:
print(new_standings.reset_index().to_markdown())

In [ ]:
predictions = pt.load_predictions(root, match_id)
old_standings = pt.load_standings_after(root, match_id - 1)
new_standings_p = pt.load_standings_after(root, match_id)
new_standings_p = new_standings.rename(columns=lambda x: x + "*")
predictions.join(old_standings, new_standings_p)

In [15]:
pt.summarize_standings_after(root, match_id)

{'TotalExacts': 5, 'TotalCorrects': 36}

# Post Template

In [16]:
m_next = pt.match_metadata(root, match_id+1)
m_last = pt.match_metadata(root, match_id)

s_pred = pt.summarize_predictions(root, match_id - 1)
s_stand = pt.summarize_standings_after(root, match_id-1)

print(f"""
# Prediction Thread Match {match_id + 2}: {m_next["HomeClub"]} vs {m_next["AwayClub"]}

# Home - Away Format

 - Home: {m_next["HomeClub"]}
 - Away: {m_next["AwayClub"]}

# Game Rules

 - Post a top level comment in this thread at least one hour before kickoff, before lineups are posted.
 - Follow the format of this example comment: "0 - 3. Banter goes here."
 - Comment format must be "[Home Score] - [Away Score]. [Optional explanation, detailed prediction, or general shit talking here.]"
 - Scoring:
    - **3 points** for an exactly correct prediction.
    - **1 points** for a correct result.
    - **0 points** for an incorrect result.
- DM me with any problems, questions, comments.

# Summary of Last Match ({m_last["HomeClub"]} - {m_last["AwayClub"]})

{m_last["ThreadUrl"]}

 - Total Predictions: {s_pred["TotalPredictions"]}
 - Predicted Home Wins: {s_pred["TotalHomeWinPredictions"]} 
 - Predicted Away Wins: {s_pred["TotalAwayWinPredictions"]} 
 - Predicted Draws: {s_pred["TotalDrawPredictions"]}
 - Exacts: {s_stand["TotalExacts"]}
 - Corrects: {s_stand["TotalCorrects"]} 

# Standings
{new_standings.reset_index().to_markdown()}
""")


# Prediction Thread Match 4: Southampton vs Manchester Utd

# Home - Away Format

 - Home: Southampton
 - Away: Manchester Utd

# Game Rules

 - Post a top level comment in this thread at least one hour before kickoff, before lineups are posted.
 - Follow the format of this example comment: "0 - 3. Banter goes here."
 - Comment format must be "[Home Score] - [Away Score]. [Optional explanation, detailed prediction, or general shit talking here.]"
 - Scoring:
    - **3 points** for an exactly correct prediction.
    - **1 points** for a correct result.
    - **0 points** for an incorrect result.
- DM me with any problems, questions, comments.

# Summary of Last Match (Brentford - Southampton)

https://www.reddit.com/r/SaintsFC/comments/1f4j1xz/prediction_thread_match_3_brentford_vs_southampton/

 - Total Predictions: 147
 - Predicted Home Wins: 101 
 - Predicted Away Wins: 18 
 - Predicted Draws: 28
 - Exacts: 3
 - Corrects: 15 

# Standings
|     | UserName             |   Points |   